In [ ]:
import numpy as np
import tensorflow as tf

In [ ]:
vocab_size = 10_000
max_length = 200
embedding_dim = 64
hidden_units = 64
batch_size = 64
epochs = 3

In [ ]:
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words = vocab_size)

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
(X_train.shape, y_train.shape), (X_test.shape, y_test.shape)

(((25000,), (25000,)), ((25000,), (25000,)))

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
X_train_padded = pad_sequences(X_train, maxlen = max_length, padding = "post", truncating = "post")
X_test_padded = pad_sequences(X_test, maxlen = max_length, padding = "post", truncating = "post")

In [ ]:
from tensorflow.keras import layers, models

In [ ]:
!pip install keras-tuner

In [ ]:
def build_rnn_model(hp):
    model = tf.keras.Sequential()

    embedding_dim = hp.Int("embedding_dim", min_value=32, max_value=128, step=32)
    model.add(layers.Embedding(input_dim=vocab_size, output_dim=embedding_dim,
                               input_shape=(max_length,)))

    rnn_type = hp.Choice("rnn_type", values=["SimpleRNN", "LSTM", "GRU"])
    rnn_units = hp.Int("rnn_units", min_value=32, max_value=128, step=32)
    dropout_rate = hp.Float("dropout_rate", min_value=0.0, max_value=0.5, step=0.1)

    if rnn_type == "SimpleRNN":
        model.add(layers.SimpleRNN(units=rnn_units, dropout=dropout_rate))
    elif rnn_type == "LSTM":
        model.add(layers.LSTM(units=rnn_units, dropout=dropout_rate))
    elif rnn_type == "GRU":
        model.add(layers.GRU(units=rnn_units, dropout=dropout_rate))

    dense_units = hp.Int("dense_units", min_value=16, max_value=64, step=16)
    model.add(layers.Dense(units=dense_units, activation="relu"))

    model.add(layers.Dense(1, activation="sigmoid"))

    learning_rate = hp.Choice("learning_rate", values=[1e-2, 1e-3, 1e-4])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [ ]:
import keras_tuner as kt

In [ ]:
tuner = kt.RandomSearch(
    hypermodel = build_rnn_model,
    objective = "val_accuracy",
    max_trials = 6,
    executions_per_trial = 1,
    directory = "rnn_tuning",
    project_name = "imdb_sentiment"
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:103: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3)

In [ ]:
tuner.search(
    X_train_padded, y_train,
    epochs = 10,
    batch_size = 64,
    validation_split = 0.2,
    callbacks = [early_stop]
)

Trial 6 Complete [00h 00m 48s]
val_accuracy: 0.8367999792098999

Best val_accuracy So Far: 0.8737999796867371
Total elapsed time: 00h 04m 33s


In [ ]:
best_model = tuner.get_best_models(num_models = 1)[0]
best_model.evaluate(X_test_padded, y_test)

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


782/782 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.8499 - loss: 0.3762


[0.37616387009620667, 0.8499199748039246]

In [ ]:
best_model.save("sentiment_analysis.keras") # loss: 0.3762, accuracy: 0.8499

In [ ]:
pt_model = tf.keras.models.load_model("sentiment_analysis.keras")

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 18 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [ ]:
emb = imdb.get_word_index()

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
import re

def predict_sentiment(custom_sent, model):
    word_index = imdb.get_word_index()

    words = custom_sent.split()
    review_indexed = []

    for word in words:
        if word in word_index and word_index[word] + 3 < vocab_size:
            review_indexed.append(word_index[word] + 3)
        else:
            review_indexed.append(2)  # <UNK> token

    padded_review = pad_sequences([review_indexed], maxlen=max_length, padding="post", truncating="post")
    prediction = model.predict(padded_review, verbose=0)[0][0]

    return "Positive" if prediction >= 0.5 else "Negative"

In [ ]:
predict_sentiment(
    "This was the best movie I watched!",
    pt_model
)

'Positive'